# aiomfac_py Quickstart

Worked examples for **aiomfac_py**, a pure-Python port of [AIOMFAC](https://github.com/andizuend/AIOMFAC) (Aerosol Inorganic–Organic Mixtures Functional groups Activity Coefficients).

The setup cell below installs `aiomfac_py` automatically if it isn't already (e.g. from having run `bootstrap_colab.ipynb` first in this session).

1. Basic activity coefficients from an AIOMFAC-web input file
2. Building a mixture by hand with `Component` (a simple aqueous NaCl example)
3. The joint bisulfate + bicarbonate dissociation equilibrium (with Ca2+ precipitation)
4. Going straight from a SMILES string with `aiomfac_py.s2as` (optional, needs `INSTALL_SMILES = True` in the bootstrap notebook)

In [1]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/content/aiomfac-python")

INSTALL_SMILES = False  # <-- set True to also enable section 4 below (aiomfac_py.s2as, needs epam.indigo)

# Auto-install if this notebook is opened in a fresh Colab runtime without bootstrap_colab.ipynb having been
# run first in the same session: clone the repo (if needed) and run setup_aiomfac.py, exactly what
# bootstrap_colab.ipynb itself does. If aiomfac_py is already importable (bootstrap_colab.ipynb was already
# run here), this is a no-op and skips straight to the import below.
try:
    import aiomfac_py as aiomfac
except ImportError:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/SeoulTechPSE/aiomfac-python.git", str(REPO_DIR)],
                        check=True)
    opts = "--carbonate --smiles" if INSTALL_SMILES else "--carbonate"
    get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_aiomfac.py'} {opts}")
    import aiomfac_py as aiomfac

# aiomfac_py itself may already have been installed above (or by bootstrap_colab.ipynb) without the optional
# `smiles` extra -- handle that case separately, so flipping INSTALL_SMILES to True and re-running this cell
# is enough on its own, without needing a --clean reinstall of aiomfac_py:
if INSTALL_SMILES:
    try:
        from indigo import Indigo  # noqa: F401
    except ImportError:
        print("Installing epam.indigo for aiomfac_py.s2as ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "epam.indigo"], check=True)

print(f"aiomfac_py {aiomfac.__version__}"
      + ("  (with aiomfac_py.s2as / epam.indigo)" if INSTALL_SMILES else ""))

aiomfac_py Colab setup
Repo dir : /content/aiomfac-python
Extras   : carbonate
Editable : True
Clean    : False

Installing aiomfac_py with extras [carbonate] ...
$ /usr/bin/python3 -m pip install -q -e .[carbonate]



Verifying installation (in this process, not a subprocess)...
aiomfac_py 0.0.11  (validated against AIOMFAC-web commit b9cb96d0eb22)
  scipy 1.17.1 OK

aiomfac_py setup complete!

Next steps:
  from aiomfac_py import read_input_file, ActivityModel
  (pass --smiles above, or `pip install aiomfac_py[smiles]`, to enable
   aiomfac_py.s2as, SMILES -> AIOMFAC subgroups)
  See notebooks/01_quickstart.ipynb for worked examples.
aiomfac_py 0.0.11


## 1. Activity coefficients from an AIOMFAC-web input file

The package ships with the two example input files from the AIOMFAC repository for quick testing.

In [2]:
from aiomfac_py import read_input_file, ActivityModel

case = read_input_file(REPO_DIR / "tests/reference/inputs/input_0001.txt")
print(case.title)
for c in case.components:
    print(f"  component {c.number}: {c.name}  subgroups={c.subgroups}")

model = ActivityModel(case.components)
res = model.evaluate(case.fractions[0], case.T_K[0], case.basis)

print()
print("T_K:", res.T_K)
print("mass fractions (wtf):", res.wtf)
print("activity coefficients (neutrals):", res.gamma_neutral)
print("ionic strength (molal):", res.ionic_strength)

Input file for AIOMFAC-web model
  component 1: Water  subgroups=((16, 1),)
  component 2: 2-Butanol  subgroups=((141, 1), (145, 1), (146, 1), (151, 1), (153, 1))
  component 3: 1-Hexanol  subgroups=((145, 1), (146, 4), (150, 1), (153, 1))
  component 4: 2-Pentanol  subgroups=((141, 1), (145, 1), (146, 2), (151, 1), (153, 1))
  component 5: smiles CC(C)CCOCCO  subgroups=((1, 2), (2, 2), (3, 1), (25, 1), (150, 1), (153, 1))
  component 6: NaCl  subgroups=((202, 1), (242, 1))

T_K: 302.0
mass fractions (wtf): [0.18307236 0.15064238 0.20765561 0.179149   0.26868244 0.01079821]
activity coefficients (neutrals): [1.85780152 1.33671042 1.80517272 1.56330817 0.60670393]
ionic strength (molal): 0.1867816424725915


## 2. Building a mixture by hand: aqueous NaCl

`Component` takes an AIOMFAC subgroup number and quantity per component. Subgroup 16 is water; ions use AIOMFAC's
own numbering (cations 201–240, anions 241–265) — Na+ is 205, Cl- is 242. See the
[AIOMFAC subgroup list](https://aiomfac.lab.mcgill.ca/subgrouplist.html) for the full table, or use
`aiomfac_py.s2as` (section 4 below) to get subgroups directly from a SMILES string for organics.

In [3]:
from aiomfac_py import Component, activity_coefficients

water = Component(1, "Water", ((16, 1),))
nacl = Component(2, "NaCl", ((205, 1), (242, 1)))

# 10% NaCl by mass, at 298.15 K:
res = activity_coefficients([water, nacl], [0.9, 0.1], 298.15, basis="mass")
print("gamma (water):", res.gamma_neutral[0])
print("ionic strength (molal):", res.ionic_strength)
print("mean ionic activity-related ln(gamma):", res.ln_gamma)

gamma (water): 0.956735054588233
ionic strength (molal): 3.0473961523576176
mean ionic activity-related ln(gamma): [-0.04422878  0.32212364  0.23499983]


## 3. Joint bisulfate + bicarbonate dissociation equilibrium

Pass just the neutral SMILES-free ionic subgroups for HSO4-/HCO3- and `ActivityModel` completes the rest of the
system automatically (adding H+, SO4--, CO3--, OH-, CO2(aq) as needed) and solves the coupled 6-unknown
dissociation equilibrium (Fortran `HSO4_and_HCO3_dissociation`) — this is the piece validated to machine
precision against the instrumented Fortran model; see the package README for details.

If Ca2+ (subgroup 221) is also present, the Ca2+ + SO4-- -> CaSO4(s) precipitation pre-step runs automatically
first.

In [4]:
# Water + NaHSO4 + NaHCO3 (no Ca2+):
water = Component(1, "Water", ((16, 1),))
nahso4 = Component(2, "NaHSO4", ((202, 1), (248, 1)))
nahco3 = Component(3, "NaHCO3", ((202, 1), (250, 1)))

res = activity_coefficients([water, nahso4, nahco3], [0.9, 0.05, 0.05], 298.15, basis="mass")
print("ionic strength (molal):", res.ionic_strength)
print("ln(gamma), all species:", res.ln_gamma)

ionic strength (molal): 1.5425297911102382
ln(gamma), all species: [ 0.0084126   0.30580119 -0.71567665 -1.11453215 -0.34305574 -0.56704086
 -2.50568546 -2.33092498 -0.20486272]


In [5]:
# Now with Ca2+ present (CaCl2), triggering the CaSO4(s) precipitation pre-step:
cacl2 = Component(2, "CaCl2", ((221, 1), (242, 2)))
nahso4 = Component(3, "NaHSO4", ((202, 1), (248, 1)))
nahco3 = Component(4, "NaHCO3", ((202, 1), (250, 1)))

res = activity_coefficients([water, cacl2, nahso4, nahco3], [0.85, 0.03, 0.03, 0.03], 298.15, basis="mass")
print("ionic strength (molal):", res.ionic_strength)

ionic strength (molal): 0.7221116308619182


## 4. SMILES → AIOMFAC subgroups (`aiomfac_py.s2as`)

Requires the optional `epam.indigo` dependency: set `INSTALL_SMILES = True` in the setup cell above and re-run
it (no restart needed). This integrates the
[S2AS tool](https://github.com/andizuend/S2AS__SMILES_to_AIOMFAC) (Amaladhasan et al., 2026) so organic
components can be specified directly by SMILES instead of by hand-picked subgroup counts.

In [6]:
try:
    from aiomfac_py.s2as import smiles_to_components
except ImportError as e:
    print(e)
    print()
    print("Set INSTALL_SMILES = True in the setup cell above and re-run it (no restart needed).")
else:
    result = smiles_to_components(["CCCCO"])  # 1-butanol; water is prepended automatically
    for c in result.components:
        print(f"  component {c.number}: {c.name}  subgroups={c.subgroups}")

    model = ActivityModel(result.components)
    res = model.evaluate([0.9, 0.1], 298.15, "mass")  # 90/10 mass-fraction water/butanol
    print()
    print("gamma (water, butanol):", res.gamma_neutral)

  component 1: Water  subgroups=((16, 1),)
  component 2: CCCCO  subgroups=((145, 1), (146, 2), (150, 1), (153, 1))

gamma (water, butanol): [ 1.00905872 24.10947049]
